# Resultados del pipeline de limpieza

El notebook [`01_dirty_data_eda.ipynb`](01_dirty_data_eda.ipynb) diagnosticó los defectos de `data/raw/messy_it_tickets.csv`. Este notebook corre el pipeline completo (`src/pipeline.run_pipeline`) sobre esos mismos datos y examina el resultado: cuántas filas quedaron limpias, cuántas fueron rechazadas y por qué, y cómo se ven las transformaciones clave (fechas, monedas, nombres de empresa) antes y después.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import run_pipeline

COLOR_VALID = "#2a78d6"
COLOR_INVALID = "#eb6834"

raw_df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "messy_it_tickets.csv")
valid_df, invalid_df = run_pipeline()

print(f"Filas crudas:     {len(raw_df):,}")
print(f"Filas válidas:    {len(valid_df):,}")
print(f"Filas inválidas:  {len(invalid_df):,}")


## 1. Válidas vs. inválidas

Las filas inválidas no son un fallo del pipeline — son filas cuyo problema de calidad de datos (empresa, agente o categoría faltante) no tiene un valor correcto que inventar, así que quedan expuestas para revisión en vez de imputadas a ciegas.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Válidas", "Inválidas"], [len(valid_df), len(invalid_df)], color=[COLOR_VALID, COLOR_INVALID])
ax.set_ylabel("Filas")
ax.set_title("Resultado de la validación de esquema")
for i, v in enumerate([len(valid_df), len(invalid_df)]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


## 2. Por qué se rechaza una fila: motivos más comunes


In [ ]:
import re

reasons = []
for err in invalid_df["validation_error"]:
    for line in err.split("\n"):
        line = line.strip()
        if line and not line[0].isdigit() and "validation error" not in line \
                and "Value error" not in line and "For further" not in line and "[type=" not in line:
            reasons.append(line)

reason_counts = pd.Series(reasons).value_counts()
print(reason_counts)

fig, ax = plt.subplots(figsize=(6, 4))
reason_counts.plot(kind="barh", ax=ax, color=COLOR_INVALID)
ax.set_xlabel("Filas rechazadas")
ax.set_title("Campo que causó el rechazo")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Muestra de filas limpias

Fechas en ISO 8601 UTC, `cost` como `float`, `user_metadata` aplanado en columnas.


In [ ]:
valid_df[["ticket_id", "company_name", "created_at", "resolved_at", "cost", "response_time_hours"]].head(10)


## 4. Unificación de nombres de empresa

`rapidfuzz` colapsa las variantes/typos de cada empresa a un único nombre canónico.


In [ ]:
raw_unique = raw_df["company_name"].nunique()
clean_unique = valid_df["company_name"].nunique()
print(f"Nombres de empresa únicos — crudo: {raw_unique} -> limpio: {clean_unique}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Crudo", "Limpio"], [raw_unique, clean_unique], color=[COLOR_INVALID, COLOR_VALID])
ax.set_ylabel("Nombres de empresa únicos")
ax.set_title("Efecto de unify_similar_names (rapidfuzz)")
plt.tight_layout()
plt.show()


## 5. Distribución de `cost` por categoría (ya limpio e imputado)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
valid_df.boxplot(column="cost", by="category", ax=ax, showfliers=False)
ax.set_title("cost por categoría (imputación por media condicional aplicada)")
plt.suptitle("")
ax.set_xlabel("category")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 6. Conclusiones

- El pipeline convierte un CSV con al menos 6 formatos de fecha, monedas mixtas y cientos de variantes de nombre de empresa en un dataset tipado y consistente.
- La tasa de rechazo (columna `validation_error`) es una métrica de calidad de datos en sí misma: cuantifica cuánta información realmente no se puede recuperar automáticamente, en vez de esconderla detrás de una imputación silenciosa.
- `data/processed/invalid_it_tickets.csv` queda disponible para que un humano revise y corrija esas filas — el pipeline nunca las descarta en silencio.
